In [0]:
SELECT * FROM bronze_db.raw_trips LIMIT 10

In [0]:
CREATE DATABASE IF NOT EXISTS silver_db

In [0]:
USE silver_db

- passenger_count > 0
- trip_distance > 0
- total_amount > 0
- Date et heure de démarrage doit être antérieur à la date et heure de fin du trajet
- Considérer uniquement les paiements en carte de crédit

## Pipeline Silver Incrémental


### 1. Préparer la table Silver (si elle n'existe pas encore)

In [0]:
CREATE TABLE IF NOT EXISTS processed_trips
USING DELTA
AS SELECT * FROM bronze_db.raw_trips
WHERE passenger_count > 0
    AND trip_distance > 0
    AND total_amount > 0
    AND tpep_pickup_datetime < tpep_dropoff_datetime
    AND tip_amount >= 0
    AND payment_type = 1;

In [0]:
SELECT COUNT(*) FROM processed_trips

In [0]:
SELECT DISTINCT file_name FROM processed_trips 
ORDER BY file_name

## 2. Charger seulement les nouvelles données de la table Bronze

In [0]:
WITH new_data AS(
    SELECT * FROM bronze_db.raw_trips AS b
    WHERE b.ingestion_timestamp > (
        SELECT MAX(ingestion_timestamp) FROM processed_trips
    )
    AND b.passenger_count > 0
    AND b.trip_distance > 0
    AND b.total_amount > 0
    AND b.tpep_pickup_datetime < tpep_dropoff_datetime
    AND b.tip_amount >= 0
    AND b.payment_type = 1
)

MERGE INTO processed_trips AS s
USING new_data AS n
ON s.VendorID = n.VendorID
AND s.tpep_pickup_datetime = n.tpep_pickup_datetime
AND s.tpep_dropoff_datetime = n.tpep_dropoff_datetime
AND s.ingestion_timestamp = n.ingestion_timestamp
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

In [0]:
SELECT COUNT(*) FROM processed_trips

In [0]:
SELECT DISTINCT file_name FROM processed_trips 
ORDER BY file_name